In [1]:
import sys
from pathlib import Path
import json
import pandas as pd

if "../src" not in sys.path:
    sys.path.append("../src")

import mp2

readme_path = Path("../README.md") if Path("../README.md").exists() else Path("README.md")
out_dir = Path(mp2.OUT)

assert readme_path.exists(), f"Missing README at {readme_path.resolve()}"
readme_text = readme_path.read_text(encoding="utf-8")

results_df = pd.read_csv(out_dir / "model_results.csv")
meta = json.loads((out_dir / "model_meta.json").read_text(encoding="utf-8"))

expected_tokens = []

# 1. Key metadata numbers & rates
for key in ("n_train", "n_test"):
    if meta.get(key) is not None:
        expected_tokens.append((f"meta.{key}", str(meta[key])))

for rate_key in ("cv_pr_auc", "train_base_rate", "test_base_rate"):
    val = meta.get(rate_key)
    if isinstance(val, (int, float)):
        expected_tokens.append((f"meta.{rate_key}", f"{val:.4f}"))

# 2. Held-out test metrics per feature set
metric_cols = [c for c in ["pr_auc", "recall_pos", "precision_pos", "f1_pos", "roc_auc"] if c in results_df.columns]
for _, row in results_df.iterrows():
    fs = row.get("feature_set", "row")
    for col in metric_cols:
        if pd.notna(row[col]):
            expected_tokens.append((f"{fs}.{col}", f"{float(row[col]):.4f}"))
    for cnt_col in ("tp", "fn", "fp", "tn"):
        if cnt_col in results_df.columns and pd.notna(row[cnt_col]):
            expected_tokens.append((f"{fs}.{cnt_col}", str(int(row[cnt_col]))))

# 3. Optional TA alignment metrics if present
align_path = out_dir / "ta_cluster_alignment.json"
if align_path.exists():
    ta_sum = json.loads(align_path.read_text(encoding="utf-8"))
    for k in ("ari_vs_ta", "nmi_vs_ta"):
        if isinstance(ta_sum.get(k), (int, float)):
            expected_tokens.append((f"ta.{k}", f"{ta_sum[k]:.4f}"))

missing_in_readme = [(label, tok) for label, tok in expected_tokens if tok not in readme_text]
found_in_readme = [(label, tok) for label, tok in expected_tokens if tok in readme_text]

print(f"Checking {readme_path.resolve()} against {out_dir.resolve()}")
print(f"Matched tokens in README.md : {len(found_in_readme)} / {len(expected_tokens)}")
if missing_in_readme:
    print("\nNot found verbatim in README.md (check rounding, omitted metrics, or stale numbers):")
    for label, tok in missing_in_readme:
        print(f"  - {label:28s}: {tok}")
else:
    print("All expected artifact values appear verbatim in README.md.")

Checking C:\Users\akdux\dsai-mini-project-2\README.md against C:\Users\akdux\dsai-mini-project-2\outputs
Matched tokens in README.md : 43 / 43
All expected artifact values appear verbatim in README.md.


In [2]:
import sys
from pathlib import Path
import json
import pandas as pd

if "../src" not in sys.path:
    sys.path.append("../src")

import mp2

readme_path = Path("../README.md") if Path("../README.md").exists() else Path("README.md")
out_dir = Path(mp2.OUT)

assert readme_path.exists(), f"Missing README at {readme_path.resolve()}"
readme_text = readme_path.read_text(encoding="utf-8")

results_df = pd.read_csv(out_dir / "model_results.csv")
meta = json.loads((out_dir / "model_meta.json").read_text(encoding="utf-8"))

expected_tokens = []

# 1. Key metadata numbers & rates
for key in ("n_train", "n_test"):
    if meta.get(key) is not None:
        expected_tokens.append((f"meta.{key}", str(meta[key])))

for rate_key in ("cv_pr_auc", "train_base_rate", "test_base_rate"):
    val = meta.get(rate_key)
    if isinstance(val, (int, float)):
        expected_tokens.append((f"meta.{rate_key}", f"{val:.4f}"))

# 2. Held-out test metrics per feature set
metric_cols = [
    c
    for c in ["pr_auc", "recall_pos", "precision_pos", "f1_pos", "roc_auc"]
    if c in results_df.columns
]
for _, row in results_df.iterrows():
    fs = row.get("feature_set", "row")
    for col in metric_cols:
        if pd.notna(row[col]):
            expected_tokens.append((f"{fs}.{col}", f"{float(row[col]):.4f}"))
    for cnt_col in ("tp", "fn", "fp", "tn"):
        if cnt_col in results_df.columns and pd.notna(row[cnt_col]):
            expected_tokens.append((f"{fs}.{cnt_col}", str(int(row[cnt_col]))))

# 3. Optional TA alignment metrics if present
align_path = out_dir / "ta_cluster_alignment.json"
if align_path.exists():
    ta_sum = json.loads(align_path.read_text(encoding="utf-8"))
    for k in ("ari_vs_ta", "nmi_vs_ta"):
        if isinstance(ta_sum.get(k), (int, float)):
            expected_tokens.append((f"ta.{k}", f"{ta_sum[k]:.4f}"))

print(f"Total expected tokens: {len(expected_tokens)}\n")
for idx, (label, tok) in enumerate(expected_tokens, start=1):
    status = "MATCH  " if tok in readme_text else "MISSING"
    print(f"{idx:2d}. [{status}] {label:30s} -> {tok}")

Total expected tokens: 43

 1. [MATCH  ] meta.n_train                   -> 1712
 2. [MATCH  ] meta.n_test                    -> 571
 3. [MATCH  ] meta.cv_pr_auc                 -> 0.8767
 4. [MATCH  ] meta.train_base_rate           -> 0.2506
 5. [MATCH  ] meta.test_base_rate            -> 0.2504
 6. [MATCH  ] area profile.pr_auc            -> 0.8633
 7. [MATCH  ] area profile.recall_pos        -> 0.8531
 8. [MATCH  ] area profile.precision_pos     -> 0.7305
 9. [MATCH  ] area profile.f1_pos            -> 0.7871
10. [MATCH  ] area profile.roc_auc           -> 0.9298
11. [MATCH  ] area profile.tp                -> 122
12. [MATCH  ] area profile.fn                -> 21
13. [MATCH  ] area profile.fp                -> 45
14. [MATCH  ] area profile.tn                -> 383
15. [MATCH  ] + cluster.pr_auc               -> 0.8656
16. [MATCH  ] + cluster.recall_pos           -> 0.8462
17. [MATCH  ] + cluster.precision_pos        -> 0.7246
18. [MATCH  ] + cluster.f1_pos               -> 0.7806
19